# EE2211 Helper Functions for Exam Acceleration
This notebook acts as a quick-reference and implementation guide for various Machine Learning algorithms covered in EE2211, tailored according to the `instructions.md` plan.

In [12]:
import numpy as np
import pandas as pd
import sklearn as sk
from sklearn.preprocessing import PolynomialFeatures
from numpy.linalg import inv

print("Libraries loaded successfully.")

Libraries loaded successfully.


### 1. Data Preprocessing & Feature Selection
This module prepares raw data for machine learning models by handling missing values, scaling, and feature selection. 

**Functions included:**
*   `universal_preprocess(df, columns_to_impute, target_col, test_split)`: The ultimate preprocessor. It replaces `0`s with `NaN` in specified columns, applies mean imputation, extracts the target variable `y`, **automatically appends a bias column of 1s** to `X`, and splits the data into training and test sets [1, 2].
*   `impute_zeros_with_mean(df, columns)`: Specifically replaces `0` entries with `NaN` and fills them with the column mean. Crucial for datasets where `0` represents a missing measurement [1].
*   `select_best_feature(X, y)`: Computes the Pearson correlation coefficient ($r$) for every feature against the target. It automatically returns the feature with the **highest absolute correlation**, as strong negative correlations are just as predictive as positive ones.
*   `add_bias(X)`: Manually prepends a column of `1`s to your feature matrix. **Always apply this before running standard linear regression (degree=1)**.
*   `normalize_z_score(X)` / `normalize_min_max(X)`: Rescales data to either a mean of 0 and standard deviation of 1 (Z-score), or strictly between [3] (Min-Max) [1, 2].

In [3]:
def impute_zeros_with_mean(df, columns):
    """
    Replace 0 values in specified physiological columns with NaN,
    and then apply mean imputation for data integrity.
    """
    df_clean = df.copy()
    # 1. Replace "0" entries with NaN
    df_clean[columns] = df_clean[columns].replace(0, np.nan)
    # 2. Apply Mean Imputation
    df_clean[columns] = df_clean[columns].fillna(df_clean[columns].mean())
    return df_clean

def normalize_min_max(X):
    """Scales data into the range [0, 1]."""
    return (X - np.min(X, axis=0)) / (np.max(X, axis=0) - np.min(X, axis=0))

def normalize_z_score(X):
    """Rescales data to a mean of 0 and standard deviation of 1."""
    return (X - np.mean(X, axis=0)) / np.std(X, axis=0)

def partition_data(X, y, test_split=0.2):
    """
    Perform a random permutation of data indices before splitting.
    Validation set size will be matched to the Test set size.
    """
    N = len(X)
    indices = np.random.permutation(N)
    
    test_size = int(N * test_split)
    val_size = test_size  # Validation equal to Test size according to instructions
    
    test_idx = indices[:test_size]
    val_idx = indices[test_size : test_size + val_size]
    train_idx = indices[test_size + val_size:]
    
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx], X[test_idx], y[test_idx]


def universal_preprocess(df, columns_to_impute, target_col, test_split=0.26):
    """
    1. Replaces 0s with NaN in specified columns and applies mean imputation.
    2. Extracts features (X) and target (y).
    3. Automatically appends a bias column of 1s to X.
    4. Performs train-test split.
    """
    df_clean = df.copy()
    
    # Imputation
    df_clean[columns_to_impute] = df_clean[columns_to_impute].replace(0, np.nan)
    df_clean[columns_to_impute] = df_clean[columns_to_impute].fillna(df_clean[columns_to_impute].mean())
    
    # Separate X and y
    y = df_clean[target_col].values
    X_raw = df_clean.drop(target_col, axis=1).values
    
    # Add bias column (offset)
    x0 = np.ones((len(y), 1))
    X_bias = np.hstack((x0, X_raw))
    
    # Train-test split
    X_train, X_test, y_train, y_test = sk.model_selection.train_test_split(X_bias, y, test_size=test_split, random_state=0)
    
    return X_train, X_test, y_train, y_test

### 2. Universal Regression Solver
This section handles parameter estimation for Ordinary Least Squares (Linear), Polynomial, and Ridge Regression [4, 5]. 

**Functions included:**
*   `solve_universal_regression(X_train, y_train, X_test, degree, lambda_val)`: Automatically solves any regression problem [5]. 
    *   **Polynomial Handling:** If `degree > 1`, it generates polynomial matrices using `fit_transform` for training data and strictly `transform` for testing data to prevent data leakage [5]. *(Note: PolynomialFeatures adds the bias column automatically, so do not use `add_bias` first).*
    *   **Matrix Dimensions:** It checks if the matrix is over-determined ($m > d$) or under-determined ($m < d$). If $m > d$, it computes the **Primal Form** (Left-Inverse). If $m < d$, it computes the **Dual Form** (Right-Inverse) [4, 5].
    *   **Regularization:** Pass a value $> 0$ to `lambda_val` to apply Ridge Regression (L2 penalty) [5].
*   `solve_regression(X, y, lambda_val)`: A leaner base function that computes optimal weights without polynomial or test-set transformations [2, 4].

In [29]:
def add_bias(X):
    """Prepends a column of 1s to matrix X for degree=1 regression."""
    return np.hstack((np.ones((len(X), 1)), X))

def solve_regression(X, y, lambda_val=0.0):
    """
    Calculates optimal weights for Primal/Dual forms & Ridge Regression.
    X: Feature matrix of shape (N, P), where N = samples, P = parameters
    """
    N, P = X.shape
    
    if lambda_val > 0:
        # Lambda-Positive Solver (Ridge Regression) - Primal Form
        I = np.eye(P)
        w = np.linalg.inv(X.T @ X + lambda_val * I) @ X.T @ y
    else:
        # Lambda = 0
        if N > P:
            # Primal Solution (Over-determined)
            w = np.linalg.inv(X.T @ X) @ X.T @ y
        else:
            # Dual Solution (Under-determined)
            w = X.T @ np.linalg.inv(X @ X.T) @ y
            
    return w

def solve_universal_regression(X_train, y_train, X_test, degree=1, lambda_val=0.0):
    """
    Solves Linear, Ridge, or Polynomial regression automatically using Primal or Dual forms.
    NOTE: If degree > 1, pass the RAW X matrices (without manual bias), as PolynomialFeatures adds the bias automatically.
    """
    # Polynomial Generation
    if degree > 1:
        poly = PolynomialFeatures(degree)
        X_train = poly.fit_transform(X_train) # Strictly fit_transform for training
        X_test = poly.transform(X_test)       # Strictly transform for testing
        
    m, d = X_train.shape
    
    # Dual vs Primal branching based on matrix dimensions
    if m > d: 
        # Over-determined: Use Left-Inverse / Primal Form
        identity = np.identity(d)
        w = inv(X_train.T @ X_train + lambda_val * identity) @ X_train.T @ y_train
    else: 
        # Under-determined: Use Right-Inverse / Dual Form
        identity = np.identity(m)
        w = X_train.T @ inv(X_train @ X_train.T + lambda_val * identity) @ y_train
        
    y_pred = X_test @ w
    
    # Calculate parameter count (useful for MCQ)
    # Excludes the bias column from d for the formula calculation if it was manually added earlier
    d_original = d - 1 if degree == 1 else d 
    
    return w, y_pred

def solve_regression_weights(X, y, degree=1, lambda_val=0.0):
    """
    Calculates the optimal weight vector 'w' for Linear, Polynomial, and Ridge Regression.
    Automatically handles Primal (Left-Inverse) vs. Dual (Right-Inverse) logic based on dimensions.
    
    IMPORTANT: If degree=1, ensure you use the `add_bias(X)` function on X before passing it here.
    If degree > 1, pass the raw X, as PolynomialFeatures automatically adds the bias column.
    """
    # 1. Polynomial Expansion
    if degree > 1:
        poly = PolynomialFeatures(degree)
        X_calc = poly.fit_transform(X)
    else:
        X_calc = X 
        
    m, d = X_calc.shape
    
    # 2. Weight Calculation based on Regularization and Dimensions
    if lambda_val > 0:
        if m > d: 
            # Over-determined: Primal Form (Ridge)
            identity = np.identity(d)
            w = inv(X_calc.T @ X_calc + lambda_val * identity) @ X_calc.T @ y
        else: 
            # Under-determined: Dual Form (Ridge)
            identity = np.identity(m)
            w = X_calc.T @ inv(X_calc @ X_calc.T + lambda_val * identity) @ y
    else:
        if m > d:
            # Over-determined: Left-Inverse (Least-Squares Solution)
            w = inv(X_calc.T @ X_calc) @ X_calc.T @ y
        elif m < d:
            # Under-determined: Right-Inverse (Least-Norm Solution)
            w = X_calc.T @ inv(X_calc @ X_calc.T) @ y
        else:
            # Even-determined: Standard Inverse
            w = inv(X_calc) @ y
            
    return w

### 3. Classification Solver
This module adapts linear regression equations to solve binary and multi-category classification tasks [3, 6, 7].

**Functions included:**
*   `one_hot_encode(labels, num_classes)`: Transforms a 1D array of categorical labels into a One-Hot Encoded matrix [7, 8]. This is required to predict multiple classes using the `solve_universal_regression` function [6]. 
*   `predict_class(y_pred_vector)`: Used for multi-category classification. It applies an `argmax` function across the raw prediction probabilities, returning the column index of the maximum value as the strict predicted class label [7, 8].
*   **For Binary Classification:** Use the regression solver to get continuous predictions $\hat{y}$, and pass them through `np.sign(y_predict)` to threshold predictions into strict `+1` or `-1` class labels [9].

In [17]:
def one_hot_encode(labels, num_classes):
    """
    Transforms a 1D categorical array to a One-Hot Encoded matrix.
    """
    return np.eye(num_classes)[labels-1]

def predict_class(y_pred_vector):
    """
    Multi-Category (Argmax): Predicted class is index of maximum value.
    """
    return np.argmax(y_pred_vector, axis=1)

In [28]:
y = one_hot_encode(np.array([1,1,2,3,3]), 3)

X = np.array([[1,3,-2],[-4,0,-1], [3,1,8], [2,1,6], [8,4,6]])
X = add_bias(X)
w, y_pred= solve_universal_regression(X_train=X,y_train=y, X_test=np.array([1, 1,-2,4]), degree=1, lambda_val=0)

print(y_pred)

[ -1.03266332 -10.09798995  12.13065327]


### 4. Gradient Descent Step Tracer
Tracks the step-by-step intermediate weight values during gradient descent [7, 8].

**Functions included:**
*   `gradient_descent_update(w_old, gradient, learning_rate)`: A tracer for step-by-step update logic [7, 8]. Loop this function using the rule $w_{new} = w_{old} - \eta \nabla J(w_{old})$ to print intermediate weights at every iteration, which is useful for "What is the weight after 2 iterations?" questions [14].

In [6]:
def gradient_descent_update(w_old, gradient, learning_rate):
    """
    Tracer for step-by-step update logic.
    w_new = w_old - LR * Gradient
    """
    return w_old - learning_rate * gradient

### 6. Decision Tree Split Evaluator (Regression)
Automates the tedious process of calculating optimal splits and MSE for Decision Tree branches [8].

**Functions included:**
*   `find_optimal_tree_split(x_feature, y_target)`: Automatically sorts the target array based on the values of the feature array. It tests every midpoint threshold between sorted values, dividing the data into left and right nodes. It returns the exact split threshold that minimizes the Weighted Average MSE.
*   `calc_weighted_mse(n_left, mse_left, n_right, mse_right)`: Calculates the weighted Mean Squared Error using the formula $(P_{left} \cdot MSE_{left}) + (P_{right} \cdot MSE_{right})$ [8].

In [7]:
def calc_weighted_mse(n_left, mse_left, n_right, mse_right):
    """Calculates weighted MSE for evaluating a threshold split."""
    n_total = n_left + n_right
    return (n_left / n_total) * mse_left + (n_right / n_total) * mse_right

def find_optimal_tree_split(x_feature, y_target):
    """Finds the split threshold minimizing Weighted MSE."""
    sorted_indices = np.argsort(x_feature)
    x_sorted, y_sorted = x_feature[sorted_indices], y_target[sorted_indices]
    best_threshold, min_mse = None, float('inf')
    
    for i in range(1, len(x_sorted)):
        threshold = (x_sorted[i-1] + x_sorted[i]) / 2.0
        left_y, right_y = y_sorted[:i], y_sorted[i:]
        
        mse_left = np.mean((left_y - np.mean(left_y))**2) if len(left_y) > 0 else 0
        mse_right = np.mean((right_y - np.mean(right_y))**2) if len(right_y) > 0 else 0
        
        weighted_mse = calc_weighted_mse(len(left_y), mse_left, len(right_y), mse_right)
        if weighted_mse < min_mse:
            min_mse, best_threshold = weighted_mse, threshold
            
    print(f"Best Threshold: {best_threshold} | Min Weighted MSE: {min_mse}")
    return best_threshold, min_mse

In [ ]:
#Q37:

x = np.array([-4.2, -3.1, -1.8, -0.5, 0.3, 1.2, 2.4, 3.0, 3.8, 4.5, 5.1, 6.3, 7.0, 8.2, 8.7])
y = np.array([1.2, 2.5, 3.8, 4.1, 5.0, 6.3, 7.2, 8.5, 9.0, 10.2, 11.0, 12.5, 13.8, 15.0, 15.7])

best_threshold, min_mse = find_optimal_tree_split(x,y)



Best Threshold: 2.7 | Min Weighted MSE: 5.179916666666667


### 7. Unsupervised Learning (K-Means)
Provides quick calculations for K-Means clustering steps without running the full scikit-learn model [10, 15].

**Functions included:**
*   `kmeans_single_step(X, centroids)`: Used to track centroid updates iteratively. 
    1. **Assignment:** Calculates the Euclidean distance from all points to the provided centroids and assigns each point to the closest one [10, 15].
    2. **Update:** Recalculates and returns the new coordinates for each centroid as the geometric mean of its assigned data points [10, 15]. Useful for step-tracing questions.

In [8]:
def kmeans_single_step(X, centroids):
    """
    1. Assigns samples to nearest centroid (Euclidean distance).
    2. Re-calculates centroid as mean of assigned samples.
    """
    # Step 1: Assignment
    distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
    labels = np.argmin(distances, axis=1)
    
    # Step 2: Update Centroids
    new_centroids = np.zeros_like(centroids)
    for k in range(len(centroids)):
        if np.any(labels == k):
            new_centroids[k] = X[labels == k].mean(axis=0)
        else:
            new_centroids[k] = centroids[k] # Fallback if empty
            
    return labels, new_centroids

### 8. Cost Evaluation & Classification Metrics
Instantly evaluates regression cost objectives and extracts advanced classification metrics [10-12].

**Functions included:**
*   `calc_exact_cost(X, y, w, lambda_val)`: Calculates the exact Ridge Cost Function $C(w) = SSE + \lambda w^T w$. The EE2211 course explicitly defines the regularization objective using the Sum of Squared Errors (SSE), not MSE [13].
*   `evaluate_classification_metrics(y_actual, y_predicted, positive_class, cost_matrix)`: Compares actual labels to predicted labels to generate the Confusion Matrix (True Positives, True Negatives, False Positives, False Negatives) [11]. It outputs Accuracy, Precision, Recall/TPR, and FPR [11, 12]. If a custom `cost_matrix` dictionary is passed, it calculates the total financial penalty of the model [10].
*   `evaluate_cost(confusion_matrix, cost_matrix)` & `evaluate_accuracy(tp, tn, total_samples)`: Base functions for specific metric calculation [10].

In [9]:
def evaluate_cost(confusion_matrix, cost_matrix):
    """
    Calculates the total penalty based on frequency of errors multiplied by their respective penalty.
    """
    return np.sum(confusion_matrix * cost_matrix)

def evaluate_accuracy(tp, tn, total_samples):
    """Standard accuracy metric"""
    return (tp + tn) / total_samples

def calc_exact_cost(X, y, w, lambda_val=0.0):
    """
    Calculates the strict SSE Cost Function: C(w) = SSE + L2 Penalty.
    Also returns MSE just in case it is asked.
    """
    y_pred = X @ w
    sse = np.sum((y_pred - y) ** 2)
    penalty = lambda_val * np.sum(w ** 2) # lambda * w^T * w
    
    cost = sse + penalty
    mse = sse / len(y)
    
    print(f"Total Cost C(w): {cost}")
    print(f"SSE: {sse} | MSE: {mse}")
    return cost, sse, mse

def evaluate_classification_metrics(y_actual, y_predicted, positive_class=1, cost_matrix=None):
    """
    Computes all classification metrics and financial penalty (if cost matrix provided).
    Assumes positive_class is 1, and everything else is negative.
    """
    TP = np.sum((y_actual == positive_class) & (y_predicted == positive_class))
    TN = np.sum((y_actual != positive_class) & (y_predicted != positive_class))
    FP = np.sum((y_actual != positive_class) & (y_predicted == positive_class))
    FN = np.sum((y_actual == positive_class) & (y_predicted != positive_class))
    
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall_tpr = TP / (TP + FN) if (TP + FN) > 0 else 0 # True Positive Rate
    fpr = FP / (FP + TN) if (FP + TN) > 0 else 0 # False Positive Rate
    
    total_cost = 0
    if cost_matrix:
        total_cost = (cost_matrix['tp'] * TP + cost_matrix['fn'] * FN + 
                      cost_matrix['fp'] * FP + cost_matrix['tn'] * TN)
                      
    print(f"TP: {TP}, TN: {TN}, FP: {FP}, FN: {FN}")
    print(f"Accuracy: {accuracy:.4f}, Precision: {precision:.4f}")
    print(f"TPR/Recall: {recall_tpr:.4f}, FPR: {fpr:.4f}")
    if cost_matrix: print(f"Total Cost Penalty: {total_cost}")
        
    return TP, TN, FP, FN, accuracy, precision, recall_tpr, fpr, total_cost

In [40]:
from scipy.stats import pearsonr


def select_best_feature(X, y):
    """
    Calculates Pearson's r for each feature column against y.
    Returns the index of the best feature based on the highest absolute correlation.
    """
    # 1. Convert Pandas DataFrames/Series to NumPy arrays to avoid indexing errors
    if isinstance(X, (pd.DataFrame, pd.Series)):
        X = X.values
    if isinstance(y, (pd.DataFrame, pd.Series)):
        y = y.values.flatten()
        
    # 2. Prevent "tuple index out of range" by reshaping 1D arrays to 2D
    if len(X.shape) == 1:
        X = X.reshape(-1, 1)
        
    best_idx = -1
    max_abs_corr = -1
    correlations = []
    
    # 3. Safely loop through the columns using X.shape[1]
    for i in range(X.shape[1]):
        corr, _ = pearsonr(X[:, i], y)
        correlations.append(corr)
        
        # Absolute correlation comparison (as strong negative correlation is just as predictive)
        if abs(corr) > max_abs_corr:
            max_abs_corr = abs(corr)
            best_idx = i
            
    print(f"All correlations: {correlations}")
    print(f"Best feature index: {best_idx} (Absolute Correlation: {max_abs_corr})")
    return best_idx, max_abs_corr

In [35]:
#Question 1: 

x = np.array([4,7,10,2,3,9]).reshape(-1,1)
y = np.array([-1,-1,-1,1,1,1])

# x = add_bias(x)

w = solve_regression_weights(x,y,degree=4)
# print(np.array([1,6])@w)


test = np.array([4,7,10,2,3,6]).reshape(-1,1)
w,y_pred = solve_universal_regression(X_train=x, y_train=y, X_test=test, degree=4)

print(y_pred)


[-0.76338028 -1.11830986 -1.04225352  1.07605634  0.74647887 -2.11975855]


In [42]:
# Question 2:

X = np.array([[3.3459, 2.7435, -1.7253], [1.0893, 2.9113, -0.7804], [3.2103, 1.4706, -0.9944], [1.744, 1.2895, 0.5307], [1.6762, 2.1366, -1.0502]])
y = np.array([2.9972, 1.1399, 2.228, 0.3387, 2.5042])

best_id, max_corr = select_best_feature(X, y)



All correlations: [np.float64(0.6509901032456192), np.float64(0.37221827010441094), np.float64(-0.9308131481041172)]
Best feature index: 2 (Absolute Correlation: 0.9308131481041172)


In [62]:
#Question 1:

X = np.array([[3,8,3,7,2], [7,2,2,4,8], [5,6,10,2,5], [8,1,5,6,7], [4,7,4,5,3]])
y = np.array([[30,21], [92,48], [67,64], [105,82], [41,30]])

y1 = y[:,0]
y2 = y[:,1]

best_id_1, max_corr_1 = select_best_feature(X,y1)

best_id_2, max_corr_2 = select_best_feature(X,y2)

X_sel = np.column_stack((X[:,0], X[:,1], X[:,4]))
X_bias = add_bias(X_sel)

w = solve_regression_weights(X_bias, y, degree=1)

print(w)

print("y_pred: ")
y_pred = X_bias@w
print(X_bias@w)

MSE = 0
for i in range(5):
    MSE += (y_pred[i][1] - y[i][1])**2

MSE = MSE / 5

print(MSE)

All correlations: [np.float64(0.9924475403682742), np.float64(-0.9761500942530978), np.float64(0.06257372399058315), np.float64(-0.24721045482335802), np.float64(0.9600875332492824)]
Best feature index: 0 (Absolute Correlation: 0.9924475403682742)
All correlations: [np.float64(0.8361740200878754), np.float64(-0.7638860632930827), np.float64(0.4952312189993291), np.float64(-0.3511368359562555), np.float64(0.7315053400619074)]
Best feature index: 0 (Absolute Correlation: 0.8361740200878754)
[[ -78.8   -363.625]
 [  19.6     54.75 ]
 [   5.2     27.625]
 [   3.      -3.125]]
y_pred: 
[[ 27.6    15.375]
 [ 92.8    49.875]
 [ 65.4    60.25 ]
 [104.2    80.125]
 [ 45.     39.375]]
28.124999999999982


In [60]:
w, y_pred = solve_universal_regression(X_train=X_sel, y_train=y, degree=2, X_test=X_sel, lambda_val=0.1)

print(y_pred)

print("MSE:")

MSE = 0
for i in range(5):
    MSE += (y_pred[i][1] - y[i][1])**2

MSE = MSE / 5

print(MSE)


[[ 29.95339753  20.84158052]
 [ 92.02796443  48.09035913]
 [ 66.93195214  63.77391023]
 [104.98279423  81.94715425]
 [ 41.10833951  30.3654696 ]]
MSE:
0.044147757400461805


In [70]:
w, y_pred = solve_universal_regression(X_train=X_sel, y_train=y, degree=2, X_test=np.array([9,4,5]).reshape(1,-1), lambda_val=0.1)
print(y_pred)

[[116.63880591  90.87239562]]
